In [5]:
import os
import gc
import random
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

!pip install rouge-score -q

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    gc.collect()
    torch.cuda.empty_cache()

Device: cuda
GPU: Tesla T4


In [4]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=633ca8d1037ed4615ef83e753063ea30cb35ccaefb34232dcf6ca2c0710cfb42
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [6]:
!unzip -q rico_dataset_v0.1_semantic_annotations.zip -d dataset

IMAGE_DIR = '/content/dataset/rico_dataset_v0.1_semantic_annotations/semantic_annotations'
print('Images found:', len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]))

Images found: 66261


In [7]:
# Upload generated_captions_full.csv when prompted
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('/content/final_generated_captions_full.csv')
df = df.dropna()
df = df[df['caption'].str.strip().str.len() > 10]
df = df.reset_index(drop=True)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

Saving final_generated_captions_full.csv to final_generated_captions_full (1).csv
Shape: (59676, 2)
Columns: ['image', 'caption']


,image,caption
0,10.png,"A list screen displaying Learn, Every second m..."
1,100.png,"A screen displaying Log In, LOG IN WITH YOUR E..."
2,1000.png,"A screen displaying Pay bills, Quick resend."


In [19]:
# ============================================================
# CORRECT — use ALL rows, never sample after splitting
# ============================================================
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')

Train : 47,740
Val   : 5,968
Test  : 5,968


In [41]:
counter = Counter()
for caption in train_df['caption']:
    counter.update(str(caption).lower().split())

vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
for word, count in counter.items():
    if count >= 2:
        vocab[word] = len(vocab)

idx2word = {v: k for k, v in vocab.items()}
print(f'Vocabulary size: {len(vocab):,}')

Vocabulary size: 52,456


In [54]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

MAX_CAP_LEN = 30   # cap caption length — stops huge padded sequences
BATCH_SIZE  = 16

class UICaptionDataset(Dataset):
    def __init__(self, dataframe, vocab, image_dir, transform):
        self.df        = dataframe.reset_index(drop=True)
        self.vocab     = vocab
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        image  = Image.open(os.path.join(self.image_dir, row['image'])).convert('RGB')
        image  = self.transform(image)
        words  = str(row['caption']).lower().split()[:MAX_CAP_LEN]  # TRUNCATE HERE
        tokens = [1] + [self.vocab.get(w, 3) for w in words] + [2]
        return image, torch.tensor(tokens, dtype=torch.long)


def collate_fn(batch):
    images, captions = zip(*batch)
    images   = torch.stack(images)
    captions = pad_sequence(captions, batch_first=True, padding_value=0)
    return images, captions


train_dataset = UICaptionDataset(train_df, vocab, IMAGE_DIR, transform)
val_dataset   = UICaptionDataset(val_df,   vocab, IMAGE_DIR, transform)
test_dataset  = UICaptionDataset(test_df,  vocab, IMAGE_DIR, transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader):,}')
print(f'Val   batches: {len(val_loader):,}')
print(f'Test  batches: {len(test_loader):,}')

# Verify caption lengths are now controlled
imgs, caps = next(iter(train_loader))
print('Image shape  :', imgs.shape)
print('Caption shape:', caps.shape)  # should be (16, ~32) not (16, 200+)

# Sanity check
imgs, caps = next(iter(train_loader))
print('Image shape  :', imgs.shape)
print('Caption shape:', caps.shape)

Train batches: 2,984
Val   batches: 373
Test  batches: 373
Image shape  : torch.Size([16, 3, 224, 224])
Caption shape: torch.Size([16, 32])
Image shape  : torch.Size([16, 3, 224, 224])
Caption shape: torch.Size([16, 32])


In [55]:
gc.collect()
torch.cuda.empty_cache()
print(f'Free memory: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB')

Free memory: 3.01 GB


In [56]:

class EncoderCNN(nn.Module):
    def __init__(self, embed_size=256):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Freeze all layers
        for p in backbone.parameters():
            p.requires_grad = False
        # Unfreeze last block only
        for p in backbone.layer4.parameters():
            p.requires_grad = True
        self.resnet = nn.Sequential(*list(backbone.children())[:-1])
        self.linear = nn.Sequential(
            nn.Linear(512, embed_size),  # ResNet18 outputs 512, not 2048
            nn.ReLU(),
            nn.Dropout(0.3)
        )

    def forward(self, images):
        f = self.resnet(images)
        f = f.view(f.size(0), -1)
        return self.linear(f)


class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm      = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_size, vocab_size)

    def forward(self, features, captions):
        emb      = self.embedding(captions[:, :-1])
        features = features.unsqueeze(1)
        emb      = torch.cat((features, emb), dim=1)
        out, _   = self.lstm(emb)
        out      = self.dropout(out)
        return self.fc(out)


encoder = EncoderCNN(embed_size=256).to(device)
decoder = DecoderRNN(embed_size=256, hidden_size=512, vocab_size=len(vocab)).to(device)

enc_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
dec_params = sum(p.numel() for p in decoder.parameters() if p.requires_grad)
print(f'Encoder trainable params: {enc_params:,}')
print(f'Decoder trainable params: {dec_params:,}')

Encoder trainable params: 8,525,056
Decoder trainable params: 41,915,624


In [ ]:
EPOCHS     = 10
LR         = 1e-3
PATIENCE   = 3
CKPT       = 'baseline_best.pth'
ACCUM_STEPS = 2   # effective batch = 16 * 2 = 32

criterion = nn.CrossEntropyLoss(ignore_index=0)
params    = list(filter(lambda p: p.requires_grad, encoder.parameters())) + \
            list(decoder.parameters())
optimizer = torch.optim.Adam(params, lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

train_losses, val_losses = [], []
best_val_loss = float('inf')
no_improve    = 0

for epoch in range(1, EPOCHS + 1):

    # ---- Train ----
    encoder.train()
    decoder.train()
    total_train = 0
    optimizer.zero_grad()

    loop = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]')
    for batch_idx, (images, captions) in enumerate(loop):
        images   = images.to(device)
        captions = captions.to(device).long()

        features = encoder(images)
        logits   = decoder(features, captions)

        targets  = captions[:, 1:].reshape(-1)
        logits   = logits[:, :captions[:, 1:].shape[1], :]
        logits   = logits.reshape(-1, logits.shape[-1])

        loss = criterion(logits, targets) / ACCUM_STEPS
        loss.backward()

        if (batch_idx + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(params, max_norm=5.0)
            optimizer.step()
            optimizer.zero_grad()

        total_train += loss.item() * ACCUM_STEPS
        loop.set_postfix(loss=f'{loss.item() * ACCUM_STEPS:.4f}')

        # Clear cache every 50 batches
        if batch_idx % 50 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    avg_train = total_train / len(train_loader)
    train_losses.append(avg_train)

    # ---- Validate ----
    encoder.eval()
    decoder.eval()
    total_val = 0

    with torch.no_grad():
        for images, captions in val_loader:
            images   = images.to(device)
            captions = captions.to(device).long()
            features = encoder(images)
            logits   = decoder(features, captions)

            targets  = captions[:, 1:].reshape(-1)
            logits   = logits[:, :captions[:, 1:].shape[1], :]
            logits   = logits.reshape(-1, logits.shape[-1])

            total_val += criterion(logits, targets).item()

    avg_val = total_val / len(val_loader)
    val_losses.append(avg_val)
    scheduler.step(avg_val)

    print(f'Epoch {epoch:02d} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        no_improve    = 0
        torch.save({
            'encoder': encoder.state_dict(),
            'decoder': decoder.state_dict()
        }, CKPT)
        print(f'  ✓ Best model saved (val_loss={best_val_loss:.4f})')
    else:
        no_improve += 1
        print(f'  No improvement ({no_improve}/{PATIENCE})')
        if no_improve >= PATIENCE:
            print('Early stopping.')
            break

print('Training complete.')

Epoch 1/10 [Train]: 100%|██████████| 2984/2984 [49:15<00:00,  1.01it/s, loss=5.1216]


Epoch 01 | Train Loss: 6.0349 | Val Loss: 5.2434
  ✓ Best model saved (val_loss=5.2434)


Epoch 2/10 [Train]: 100%|██████████| 2984/2984 [50:39<00:00,  1.02s/it, loss=3.9053]


Epoch 02 | Train Loss: 5.1059 | Val Loss: 4.8427
  ✓ Best model saved (val_loss=4.8427)


Epoch 3/10 [Train]: 100%|██████████| 2984/2984 [48:45<00:00,  1.02it/s, loss=3.9655]


...

Epoch 4/10 [Train]: 100%|██████████| 2984/2984 [48:17<00:00,  1.03it/s, loss=4.2346]


Epoch 04 | Train Loss: 3.8163 | Val Loss: 4.5428
  ✓ Best model saved (val_loss=4.5428)


Epoch 5/10 [Train]: 100%|██████████| 2984/2984 [51:36<00:00,  1.04s/it, loss=4.0631]


Epoch 05 | Train Loss: 3.4073 | Val Loss: 4.5062
  ✓ Best model saved (val_loss=4.5062)


Epoch 6/10 [Train]:  47%|████▋     | 1395/2984 [23:28<28:15,  1.07s/it, loss=3.2550]

In [1]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses,   label='Val Loss',   marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Baseline CNN-LSTM — Training Curves')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('baseline_training_curves.png', dpi=150)
plt.show()

NameError: name 'train_losses' is not defined

<Figure size 800x400 with 0 Axes>

In [ ]:
checkpoint = torch.load(CKPT, map_location=device)
encoder.load_state_dict(checkpoint['encoder'])
decoder.load_state_dict(checkpoint['decoder'])
encoder.eval()
decoder.eval()
print('Best checkpoint loaded.')

In [ ]:
def greedy_decode(image, max_len=30):
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        image    = image.unsqueeze(0).to(device)
        features = encoder(image)
        inputs   = features.unsqueeze(1)
        hidden   = None
        result   = []

        for _ in range(max_len):
            out, hidden  = decoder.lstm(inputs, hidden)
            logits       = decoder.fc(decoder.dropout(out.squeeze(1)))
            token_id     = logits.argmax(1).item()

            if token_id == vocab['<EOS>']:
                break

            result.append(idx2word.get(token_id, '<UNK>'))
            inputs = decoder.embedding(
                torch.tensor([[token_id]], device=device)
            )

    return ' '.join(result)

In [ ]:
def decode_gt(caption_tensor):
    words = []
    for tok in caption_tensor.numpy():
        word = idx2word.get(int(tok), '')
        if word in ('<SOS>', '<EOS>', '<PAD>', ''):
            continue
        words.append(word)
    return ' '.join(words)


print('=== Qualitative Examples ===')
for i in range(10):
    idx   = random.randint(0, len(test_dataset) - 1)
    image, caption = test_dataset[idx]
    gt   = decode_gt(caption)
    pred = greedy_decode(image)
    print(f'\n--- Example {i+1} ---')
    print(f'Ground Truth : {gt}')
    print(f'Prediction   : {pred}')

In [ ]:
from google.colab import files

torch.save({
    'encoder'     : encoder.state_dict(),
    'decoder'     : decoder.state_dict(),
    'vocab'       : vocab,
    'idx2word'    : idx2word,
    'train_losses': train_losses,
    'val_losses'  : val_losses,
}, 'baseline_complete.pth')

files.download('baseline_complete.pth')
files.download('baseline_training_curves.png')
print('Done.')